# Aim of this code is to use rigid registration for stacking the 222 brain. 

In [ ]:
# STEPS:
# 1) Read the images
# 2) Resize the image (if required)
# 3) Extract the blue channel
# 4) Sort the jpg images based on the file name
# 5) Downsample the image by 2 or 3 as the final .nrrd file is consuming 2.73GB space
# 6) Compute the centroid and align it to avoid rotation of images. 
# 7) Do rigid registration of images
# 8) Save the file as .nrrd - ensure 8-bit unsigned char format while saving

In [ ]:
import os
import re
import cv2
import numpy as np
import SimpleITK as sitk
import nrrd
from tqdm import tqdm


In [ ]:
# Define the input folder containing JPG images
input_folder = '/home/projects/registration/Data/BFI_registration/222/BFI/fiducial_removed_222/222/BFI'


In [ ]:
# Set the final desired size to 1000x1000
final_height = 1000
final_width = 1000


In [ ]:
# Function to extract the image number from the filename
def extract_image_number(filename):
    pattern = re.compile(r'_(\d+)_original\.jpg$')
    match = pattern.search(filename)
    if match:
        return int(match.group(1))
    else:
        print(f"Warning: No number found in filename {filename}. Skipping.")
        return None


In [ ]:
# List and sort all JPG images in the folder based on the extracted image number
image_files = [f for f in os.listdir(input_folder) if f.endswith(".jpg")]
image_files.sort(key=lambda f: extract_image_number(f) or float('inf'))


In [ ]:
# Function to read, resize (downsample by 3), and extract the blue channel from an image
def extract_blue_channel(img_path, final_width, final_height):
    image = cv2.imread(img_path)
    if image is None:
        raise ValueError(f"Unable to read image {img_path}.")
    
    # Calculate the downsampled size (3x reduction)
    downsampled_height = final_height * 3
    downsampled_width = final_width * 3
    
    # Downsample the image first by resizing to the larger dimensions
    resized_image = cv2.resize(image, (downsampled_width, downsampled_height))
    
    # Then resize to the final desired dimensions (1000x1000)
    resized_image = cv2.resize(resized_image, (final_width, final_height))
    
    # Extract the blue channel (OpenCV loads images in BGR format)
    blue_channel = resized_image[:, :, 0]
    
    # Ensure the blue channel is of dtype uint8
    blue_channel = blue_channel.astype(np.uint8)
    
    return blue_channel

In [ ]:
# Function to compute the centroid of an image
def compute_centroid(image):
    moments = cv2.moments(image)
    if moments["m00"] != 0:
        centroid_x = int(moments["m10"] / moments["m00"])
        centroid_y = int(moments["m01"] / moments["m00"])
    else:
        centroid_x, centroid_y = image.shape[1] // 2, image.shape[0] // 2
    return np.array([centroid_x, centroid_y])

In [ ]:
# Function to align the centroid of a moving image to the fixed image
def align_centroids(fixed_image, moving_image):
    fixed_centroid = compute_centroid(fixed_image)
    moving_centroid = compute_centroid(moving_image)
    
    # Calculate the translation needed to align centroids
    translation = fixed_centroid - moving_centroid
    
    # Apply the translation using an affine transformation
    translation_matrix = np.float32([[1, 0, translation[0]], [0, 1, translation[1]]])
    aligned_image = cv2.warpAffine(moving_image, translation_matrix, (moving_image.shape[1], moving_image.shape[0]))
    
    return aligned_image

In [ ]:
# Extract the blue channel of the first image (fixed image)
fixed_image_path = os.path.join(input_folder, image_files[0])
fixed_image_blue = extract_blue_channel(fixed_image_path, final_width, final_height)

# Convert the fixed image to SimpleITK format and then to float32
fixed_image_sitk = sitk.GetImageFromArray(fixed_image_blue.astype(np.float32))

# Initialize the list for registered images
registered_images = [fixed_image_blue]  # Start with the fixed image

In [ ]:
# Registration setup using SimpleITK
def register_images(fixed_image, moving_image):
    # Initialize the registration method
    registration_method = sitk.ImageRegistrationMethod()
    
    # Use mutual information metric
    registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    
    # Set interpolator
    registration_method.SetInterpolator(sitk.sitkLinear)
    
    # Optimizer setup (gradient descent)
    registration_method.SetOptimizerAsRegularStepGradientDescent(
        learningRate=2.0, minStep=1e-4, numberOfIterations=500,
        gradientMagnitudeTolerance=1e-8)
    
    # Explicitly create an Euler2DTransform for the initialization (no rotation allowed)
    initial_transform = sitk.Euler2DTransform()
    initial_transform.SetAngle(0.0)  # Lock rotation
    
    # Initialize the Euler2DTransform using the CenteredTransformInitializer
    initial_transform = sitk.CenteredTransformInitializer(
        fixed_image, moving_image, initial_transform, sitk.CenteredTransformInitializerFilter.GEOMETRY)
    
    # Set the initial transform in the registration
    registration_method.SetInitialTransform(initial_transform, inPlace=False)

    # Execute the registration (translation only, no rotation)
    final_transform = registration_method.Execute(fixed_image, moving_image)
    
    # Resample the moving image based on the computed transform
    resampled_image = sitk.Resample(moving_image, fixed_image, final_transform, sitk.sitkLinear, 0.0, moving_image.GetPixelID())
    
    return resampled_image

In [ ]:
# Register all moving images to the fixed image with progress bar
for filename in tqdm(image_files[1:], desc="Registering images", unit="image"):
    img_path = os.path.join(input_folder, filename)
    
    # Extract blue channel of the moving image
    moving_image_blue = extract_blue_channel(img_path, final_width, final_height)
    
    # Align the centroids of the moving and fixed images
    moving_image_aligned = align_centroids(fixed_image_blue, moving_image_blue)
    
    # Convert to SimpleITK image in float32
    moving_image_sitk = sitk.GetImageFromArray(moving_image_aligned.astype(np.float32))
    
    # Perform rigid registration (translation only, no rotation)
    registered_image_sitk = register_images(fixed_image_sitk, moving_image_sitk)
    
    # Convert back to NumPy array (registered image in float32)
    registered_image = sitk.GetArrayFromImage(registered_image_sitk)
    
    # Convert back to uint8 for saving
    registered_image_uint8 = np.clip(registered_image, 0, 255).astype(np.uint8)
    
    # Append the registered uint8 image to the list
    registered_images.append(registered_image_uint8)


In [ ]:
# Convert the list of registered images to a NumPy array and ensure 8-bit unsigned char format
registered_images_array = np.stack(registered_images, axis=0).astype(np.uint8)

# Save the final registered stack as a .nrrd file
nrrd.write('cent_align_RR_stack_fid_remov.nrrd', registered_images_array)

print(f"Final shape of the registered image stack: {registered_images_array.shape}")
print(f"Data type of the registered image stack: {registered_images_array.dtype}")